# PCM with Double Explanatory IRT (EIRT) --- Multilevel Hierarchical Bayesian Model

## 학습 목표

1. 설명적 IRT(EIRT)의 구조와 목적을 설명할 수 있다.
2. 피험자 속성과 문항 속성이 능력/난이도에 미치는 효과를 분석할 수 있다.
3. Stan으로 다수준(multilevel) 계층 EIRT 모델을 추정할 수 있다.
4. 표본 크기 결정의 실용적 기준을 이해할 수 있다.

---

## 1. EIRT란? (Explanatory Item Response Theory)

**EIRT**(De Boeck & Wilson, 2004)는 잠재 변수인 능력($\theta$)과 난이도($\beta$)를
**관측 가능한 외부 변수(Covariates)**로 설명하는 가장 진보된 IRT 형태입니다.

### 1.1 구조 방정식 (Structural Equations)

**피험자 측 (Person-side):**
$$\theta_n = \sum_{q=1}^{Q} \alpha_q Z_{nq} + \zeta_n, \quad \zeta_n \sim \mathcal{N}(0, \sigma_\zeta^2)$$

**문항 측 (Item-side, LLTM):**
$$\beta_i = \sum_{p=1}^{P} \gamma_p X_{ip} + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma_\epsilon^2)$$

| 기호 | 의미 |
|---|---|
| $Z_{nq}$ | 피험자 $n$의 $q$번째 속성 (성별, 국적, 학년 등) |
| $\alpha_q$ | 속성 $q$가 능력에 미치는 효과 |
| $X_{ip}$ | 문항 $i$의 $p$번째 속성 (읽기 부하, 쓰기 부하) |
| $\gamma_p$ | 문항 속성 $p$가 난이도에 미치는 효과 |
| $\zeta_n$ | 능력의 잔차 (구조 방정식으로 설명 안 되는 부분) |
| $\epsilon_i$ | 난이도의 잔차 |

### 1.2 연구 질문 예시 (Korean Language Education Context)

- "쓰기 요구도(write_demand)가 높은 문항이 더 어려운가?" ($\gamma_{write}$)
- "학년이 높을수록 능력이 높은가?" ($\alpha_{year}$)
- "국적(heritage vs. non-heritage)에 따라 능력 차이가 있는가?" ($\alpha_{country}$)
- "성별에 따른 능력 차이가 통계적으로 유의한가?" ($\alpha_{gender}$)

---

## 2. 다수준 계층 모델이 필요한 이유

특정 조건(예: 여성+특정 국가+1학년)에 해당하는 피험자가 매우 적을 수 있습니다.
**다수준 계층 모델(Multilevel Hierarchical Model)**을 사용하면:
- 소집단의 추정치가 **전체 집단의 정보를 빌려** 안정화됩니다 (Partial Pooling)
- 추정 불확실성이 적절히 표현됩니다

위 Stan 모델에서 $\zeta_n \sim \mathcal{N}(0, \sigma_\zeta)$와 $\epsilon_i \sim \mathcal{N}(0, \sigma_\epsilon)$이
계층 구조를 구현합니다.

---

## 3. 표본 크기 결정 (Sample Size Determination)

### 실용적 기준

EIRT 모델에서 충분한 표본 수는 다음 요소를 고려합니다:

1. **공변량 세포 수**: gender(2) × country(3) × year(3) = **18개 조건**
2. **세포당 최소 피험자 수**: 각 조건에 최소 10-20명 권장
3. **최소 표본 수**: 18 × 10 = 180명 (최소) ~ 18 × 20 = 360명 (권장)

**우리 시뮬레이션에서 사용할 N = 360명**  
(각 조건당 평균 20명, 계층 모델이므로 소집단 추정치도 안정적)

**참고**: 실제 연구에서는 다음 방법으로 표본 수를 결정합니다:
- Monte Carlo 시뮬레이션 기반 Power Analysis
- `pwr` 패키지 또는 G*Power 소프트웨어
- 비슷한 선행연구의 표본 수 참조


In [ ]:
import sys, os, warnings, tempfile
import matplotlib as mpl, matplotlib.font_manager as fm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmdstanpy
from scipy import stats

warnings.filterwarnings('ignore')

def setup_korean_font():
    candidates = {
        'win32':  [('C:/Windows/Fonts/malgun.ttf', 'Malgun Gothic')],
        'darwin': [('/System/Library/Fonts/AppleSDGothicNeo.ttc', 'Apple SD Gothic Neo')],
        'linux':  [('/usr/share/fonts/truetype/nanum/NanumGothic.ttf', 'NanumGothic')],
    }
    for path, name in candidates.get(sys.platform, []):
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            mpl.rcParams['font.family'] = [name, 'DejaVu Sans']
            return
    mpl.rcParams['font.family'] = ['DejaVu Sans']

setup_korean_font()
mpl.rcParams['axes.unicode_minus'] = False
np.random.seed(5101)


## 4. 시뮬레이션 데이터 생성

### 설계
- **N = 360명** (gender×country×year = 2×3×3 = 18 조건, 각 20명)
- **I = 10 문항**, **K = 4 카테고리** (0, 1, 2, 3점)
- **피험자 속성**: gender (0=여, 1=남), country (1=한국, 2=미국, 3=중국), school_year (1, 2, 3)
- **문항 속성**: read_demand (읽기 부하, 연속), write_demand (쓰기 부하, 연속)

### 참 효과 (True Effects)
- $\alpha_{gender} = 0.3$ (남성이 약간 더 높은 능력)
- $\alpha_{country2} = -0.5$ (미국 출신), $\alpha_{country3} = 0.2$ (중국 출신)
- $\alpha_{year} = 0.4$ (학년당 능력 증가)
- $\gamma_{read} = 0.5$ (읽기 부하가 높을수록 어려움)
- $\gamma_{write} = 0.8$ (쓰기 부하가 높을수록 더 어려움)


In [ ]:
# ── 표본 크기 결정 ──────────────────────────────────────────
# Conditions: gender(2) x country(3) x year(3) = 18 cells
n_per_cell = 20
N = 18 * n_per_cell   # 360 persons
I = 10                 # 10 items
K = 4                  # 4 categories (0, 1, 2, 3)

print(f"Sample size determination:")
print(f"  Conditions: 2 (gender) x 3 (country) x 3 (year) = 18 cells")
print(f"  N per cell: {n_per_cell}")
print(f"  Total N: {N}")
print(f"  Items: {I}, Categories: {K}")
print()

# ── 피험자 공변량 생성 ───────────────────────────────────────
genders, countries, years_list = [], [], []
for gender in [0, 1]:
    for country in [1, 2, 3]:
        for year in [1, 2, 3]:
            genders.extend([gender] * n_per_cell)
            countries.extend([country] * n_per_cell)
            years_list.extend([year] * n_per_cell)

gender  = np.array(genders)     # 0 or 1
country = np.array(countries)   # 1, 2, 3
year    = np.array(years_list)  # 1, 2, 3

# Design matrix Z for person covariates
# [gender, country_2_dummy, country_3_dummy, year]
Z = np.column_stack([
    gender,
    (country == 2).astype(float),  # country 2 dummy (vs country 1)
    (country == 3).astype(float),  # country 3 dummy (vs country 1)
    (year - 2).astype(float),      # year centered at 2
])
Q = Z.shape[1]

print(f"Person covariate matrix Z: {Z.shape}")
print("  Z columns: [gender, country_2, country_3, year_centered]")

# ── 문항 속성 생성 ───────────────────────────────────────────
read_demand  = np.random.uniform(0, 1, I)
write_demand = np.random.uniform(0, 1, I)

X = np.column_stack([read_demand, write_demand])
P = X.shape[1]

print(f"\nItem covariate matrix X: {X.shape}")
print(f"  X columns: [read_demand, write_demand]")
for i in range(I):
    print(f"  Item {i+1}: read={read_demand[i]:.2f}, write={write_demand[i]:.2f}")


In [ ]:
# ── 참 파라미터 설정 ────────────────────────────────────────
# Person-side effects
alpha_true = np.array([0.3, -0.5, 0.2, 0.4])   # gender, country2, country3, year
sigma_zeta_true = 0.6

# Item-side effects
gamma_true = np.array([0.5, 0.8])   # read_demand, write_demand
sigma_epsilon_true = 0.4

# Structural model
zeta_true    = np.random.normal(0, sigma_zeta_true, N)
epsilon_true = np.random.normal(0, sigma_epsilon_true, I)

theta_true = Z @ alpha_true + zeta_true
beta_raw   = X @ gamma_true + epsilon_true
beta_true  = beta_raw - beta_raw.mean()   # center for identifiability

# Step difficulties (sum-to-zero per item)
tau_raw  = np.random.normal(0, 0.8, (I, K-1))
tau_true = tau_raw - tau_raw.mean(axis=1, keepdims=True)

print("True structural effects:")
print(f"  alpha (person): gender={alpha_true[0]}, country2={alpha_true[1]}, "
      f"country3={alpha_true[2]}, year={alpha_true[3]}")
print(f"  gamma (item): read={gamma_true[0]}, write={gamma_true[1]}")
print(f"  sigma_zeta={sigma_zeta_true},  sigma_epsilon={sigma_epsilon_true}")
print(f"\ntheta: mean={theta_true.mean():.3f}, SD={theta_true.std():.3f}")
print(f"beta:  mean={beta_true.mean():.3f}, SD={beta_true.std():.3f}")


In [ ]:
# ── 응답 데이터 생성 ────────────────────────────────────────
def pcm_probs(theta, beta_i, tau_i):
    log_p = np.zeros(K)
    for k in range(1, K):
        log_p[k] = log_p[k-1] + (theta - (beta_i + tau_i[k-1]))
    log_p -= log_p.max()
    p = np.exp(log_p)
    return p / p.sum()

Y = np.zeros((N, I), dtype=int)
for n in range(N):
    for i in range(I):
        pr = pcm_probs(theta_true[n], beta_true[i], tau_true[i])
        Y[n, i] = np.random.choice(K, p=pr)

print(f"Response matrix Y: {Y.shape}")
print(f"Category counts: {np.bincount(Y.ravel())}")

# Show ability by group
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (grp, grp_name, labels) in zip(axes, [
    (gender, 'Gender', ['Female (0)', 'Male (1)']),
    (country, 'Country', ['Korea (1)', 'USA (2)', 'China (3)']),
    (year, 'School Year', ['Year 1', 'Year 2', 'Year 3']),
]):
    unique_vals = np.unique(grp)
    box_data = [theta_true[grp == v] for v in unique_vals]
    ax.boxplot(box_data, labels=labels[:len(unique_vals)], patch_artist=True,
               boxprops=dict(facecolor='lightblue', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
    ax.set_ylabel('True Ability theta')
    ax.set_title(f'Ability by {grp_name}')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

fig.suptitle('True Ability Distribution by Person Attributes', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Stan Model Code (EIRT)

In [ ]:
stan_code = 'data {\n  int<lower=1> N;              // persons\n  int<lower=1> I;              // items\n  int<lower=2> K;              // categories\n  int<lower=0> Nobs;\n  array[Nobs] int<lower=1,upper=N> nn;\n  array[Nobs] int<lower=1,upper=I> ii;\n  array[Nobs] int<lower=1,upper=K> y;\n\n  // Person-side covariates: gender(0/1), country(1-3), school_year(1-3)\n  int<lower=1> Q;                   // number of person covariates\n  matrix[N, Q] Z;                   // person covariate matrix\n\n  // Item-side covariates: read_demand, write_demand\n  int<lower=1> P;                   // number of item covariates\n  matrix[I, P] X;                   // item covariate matrix\n}\nparameters {\n  // Person-side structural model\n  vector[Q] alpha;                  // person covariate effects on ability\n  vector[N] zeta;                   // person residuals\n  real<lower=0> sigma_zeta;         // SD of person residuals\n\n  // Item-side structural model (LLTM)\n  vector[P] gamma;                  // item covariate effects on difficulty\n  vector[I] epsilon;                // item residuals\n  real<lower=0> sigma_epsilon;\n\n  // Step difficulties (sum-to-zero per item, K-1 thresholds)\n  matrix[I, K-2] tau_free;\n}\ntransformed parameters {\n  // Person abilities and item difficulties from structural model\n  vector[N] theta;\n  vector[I] beta;\n  matrix[I, K-1] tau;\n\n  // Structural equations\n  theta = Z * alpha + zeta;    // theta = sum(alpha_q * Z_nq) + zeta_n\n  beta  = X * gamma + epsilon; // beta  = sum(gamma_p * X_ip) + epsilon_i\n\n  // Sum-to-zero tau per item\n  for (i in 1:I)\n    tau[i] = append_col(tau_free[i], -sum(tau_free[i]));\n}\nmodel {\n  // Priors\n  alpha       ~ normal(0, 1);\n  zeta        ~ normal(0, sigma_zeta);\n  sigma_zeta  ~ exponential(1);\n\n  gamma       ~ normal(0, 1);\n  epsilon     ~ normal(0, sigma_epsilon);\n  sigma_epsilon ~ exponential(1);\n\n  to_vector(tau_free) ~ normal(0, 2);\n\n  // Likelihood (PCM with decomposed form)\n  for (obs in 1:Nobs) {\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[nn[obs]] - (beta[ii[obs]] + tau[ii[obs], k-1]));\n    y[obs] ~ categorical_logit(log_p);\n  }\n}\ngenerated quantities {\n  array[Nobs] int y_rep;\n  for (obs in 1:Nobs) {\n    vector[K] log_p;\n    log_p[1] = 0;\n    for (k in 2:K)\n      log_p[k] = log_p[k-1] + (theta[nn[obs]] - (beta[ii[obs]] + tau[ii[obs], k-1]));\n    y_rep[obs] = categorical_logit_rng(log_p);\n  }\n}'

nn_arr, ii_arr, y_arr = [], [], []
for n in range(N):
    for i in range(I):
        nn_arr.append(n + 1)
        ii_arr.append(i + 1)
        y_arr.append(int(Y[n, i]) + 1)

stan_data = {
    'N': N, 'I': I, 'K': K, 'Nobs': N * I,
    'nn': nn_arr, 'ii': ii_arr, 'y': y_arr,
    'Q': Q, 'Z': Z.tolist(),
    'P': P, 'X': X.tolist(),
}

tmpdir = tempfile.mkdtemp()
stan_path = os.path.join(tmpdir, 'eirt.stan')
with open(stan_path, 'w') as f:
    f.write(stan_code)

model = cmdstanpy.CmdStanModel(stan_file=stan_path)
print('EIRT Stan model compiled.')


## 6a. MAP Estimation

In [ ]:
fit_map = model.optimize(data=stan_data, seed=5101, jacobian=True)

alpha_map = fit_map.stan_variable('alpha')
gamma_map = fit_map.stan_variable('gamma')
theta_map = fit_map.stan_variable('theta')
beta_map  = fit_map.stan_variable('beta')

print("MAP structural effects:")
print(f"  alpha (true vs MAP):")
alpha_labels = ['gender', 'country2', 'country3', 'year']
for q, label in enumerate(alpha_labels):
    print(f"    {label:10s}: true={alpha_true[q]:.3f}, MAP={alpha_map[q]:.3f}")

print(f"  gamma (true vs MAP):")
gamma_labels = ['read_demand', 'write_demand']
for p, label in enumerate(gamma_labels):
    print(f"    {label:12s}: true={gamma_true[p]:.3f}, MAP={gamma_map[p]:.3f}")

print(f"\ntheta corr: {np.corrcoef(theta_true, theta_map)[0,1]:.3f}")
print(f"beta  corr: {np.corrcoef(beta_true, beta_map)[0,1]:.3f}")

# Scatter for theta and beta
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (true, est, col, title) in zip(axes, [
    (theta_true, theta_map, 'steelblue', 'Person Ability theta (MAP)'),
    (beta_true,  beta_map,  'coral',     'Item Difficulty beta (MAP)'),
]):
    ax.scatter(true, est, color=col, s=30, alpha=0.7)
    lo, hi = min(true.min(), est.min())-0.3, max(true.max(), est.max())+0.3
    ax.plot([lo, hi], [lo, hi], 'k--')
    r = np.corrcoef(true, est)[0,1]
    ax.set_xlabel('True'); ax.set_ylabel('MAP')
    ax.set_title(f'{title}  r={r:.3f}')
plt.suptitle('EIRT MAP Estimation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 6b. Full Bayesian MCMC

In [ ]:
fit = model.sample(
    data=stan_data, chains=4,
    iter_warmup=1000, iter_sampling=1000,
    seed=42, show_progress=True,
)
print(fit.diagnose())


## 7. MCMC Convergence Analysis

In [ ]:
summary = fit.summary()
alpha_s = summary[summary.index.str.startswith('alpha[')]
gamma_s = summary[summary.index.str.startswith('gamma[')]
theta_s = summary[summary.index.str.startswith('theta[')]
beta_s  = summary[summary.index.str.startswith('beta[')]

print("Convergence:")
for nm, df in [('alpha', alpha_s), ('gamma', gamma_s), ('theta', theta_s), ('beta', beta_s)]:
    ess_col = 'N_Eff' if 'N_Eff' in df.columns else 'ESS_bulk'
    print(f"  {nm:5s}: Rhat max={df['R_hat'].max():.4f}, ESS min={df[ess_col].min():.0f}")

alpha_samples = fit.stan_variable('alpha')   # (4000, Q)
gamma_samples = fit.stan_variable('gamma')   # (4000, P)

fig, axes = plt.subplots(Q+P, 2, figsize=(13, 3*(Q+P)))
chain_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
param_names  = alpha_labels + gamma_labels
param_true_v = list(alpha_true) + list(gamma_true)
all_samples  = np.hstack([alpha_samples, gamma_samples])   # (4000, Q+P)

for row in range(Q+P):
    samp = all_samples[:, row]
    chains = samp.reshape(4, 1000)
    for c in range(4):
        axes[row, 0].plot(range(1000), chains[c], color=chain_colors[c],
                          alpha=0.7, linewidth=0.6, label=f'C{c+1}' if row==0 else '')
    axes[row, 0].set_title(f'Trace: {param_names[row]} (true={param_true_v[row]:.2f})', fontsize=9)
    if row==0: axes[row, 0].legend(fontsize=7)
    axes[row, 1].hist(samp, bins=40, density=True, color='steelblue', alpha=0.6, edgecolor='white')
    axes[row, 1].axvline(samp.mean(), color='red', linestyle='--', linewidth=1.5)
    axes[row, 1].axvline(param_true_v[row], color='green', linewidth=2.2,
                          label=f'True={param_true_v[row]:.2f}')
    axes[row, 1].set_title(f'Posterior: {param_names[row]}', fontsize=9)
    axes[row, 1].legend(fontsize=7)

axes[-1, 0].set_xlabel('Iteration')
fig.suptitle('EIRT: Trace Plots for Structural Effects', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 8. 구조 효과 추정 결과 (Structural Effect Estimates)

EIRT의 핵심: $\alpha$ (피험자 속성 효과)와 $\gamma$ (문항 속성 효과)의 95% 사후 분포


In [ ]:
alpha_samples = fit.stan_variable('alpha')
gamma_samples = fit.stan_variable('gamma')
theta_samples = fit.stan_variable('theta')
beta_samples  = fit.stan_variable('beta')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Alpha: person covariate effects
ax = axes[0]
ax.set_title('Person-side Effects (alpha): 96% Posterior CI', fontsize=12)
for q, label in enumerate(alpha_labels):
    samp = alpha_samples[:, q]
    ci_lo, ci_hi = np.percentile(samp, 2), np.percentile(samp, 98)
    pm = samp.mean()
    ax.barh(q, pm, color='steelblue', alpha=0.7)
    ax.plot([ci_lo, ci_hi], [q, q], 'k-', linewidth=2)
    ax.plot(alpha_true[q], q, 'g^', markersize=12, zorder=10)
    ax.text(ci_hi + 0.02, q, f'{pm:.3f}', va='center', fontsize=9)

ax.set_yticks(range(Q))
ax.set_yticklabels(alpha_labels, fontsize=10)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Effect size (logit)')
ax.text(0.05, 0.95, 'Green triangle = true value', transform=ax.transAxes,
        fontsize=8, va='top')

# Gamma: item covariate effects
ax = axes[1]
ax.set_title('Item-side Effects (gamma): 96% Posterior CI', fontsize=12)
for p, label in enumerate(gamma_labels):
    samp = gamma_samples[:, p]
    ci_lo, ci_hi = np.percentile(samp, 2), np.percentile(samp, 98)
    pm = samp.mean()
    ax.barh(p, pm, color='coral', alpha=0.8)
    ax.plot([ci_lo, ci_hi], [p, p], 'k-', linewidth=2)
    ax.plot(gamma_true[p], p, 'g^', markersize=12, zorder=10)
    ax.text(ci_hi + 0.02, p, f'{pm:.3f}', va='center', fontsize=9)

ax.set_yticks(range(P))
ax.set_yticklabels(gamma_labels, fontsize=10)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Effect on item difficulty (logit)')

fig.suptitle('EIRT Structural Effect Estimates\n(horizontal line = 96% CI, triangle = true value)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\n=== 구조 효과 추정 결과 해석 ===")
print("alpha:")
for q, label in enumerate(alpha_labels):
    pm = alpha_samples[:, q].mean()
    ci = np.percentile(alpha_samples[:, q], [2, 98])
    sig = 'SIGNIFICANT' if (ci[0] > 0 or ci[1] < 0) else 'not significant'
    print(f"  {label:10s}: estimate={pm:.3f}, 96%CI=[{ci[0]:.3f},{ci[1]:.3f}] -> {sig}")
print("gamma:")
for p, label in enumerate(gamma_labels):
    pm = gamma_samples[:, p].mean()
    ci = np.percentile(gamma_samples[:, p], [2, 98])
    sig = 'SIGNIFICANT' if (ci[0] > 0 or ci[1] < 0) else 'not significant'
    print(f"  {label:12s}: estimate={pm:.3f}, 96%CI=[{ci[0]:.3f},{ci[1]:.3f}] -> {sig}")


## 9. Posterior Predictive Check (PPC)

In [ ]:
y_rep_samples = fit.stan_variable('y_rep')
y_flat = np.array(y_arr) - 1
obs_mean = y_flat.mean()
rep_means = (y_rep_samples - 1).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(rep_means, bins=40, density=True, color='steelblue', alpha=0.7, edgecolor='white',
             label='y_rep distribution')
axes[0].axvline(obs_mean, color='red', linewidth=2.5, label=f'Observed={obs_mean:.3f}')
axes[0].set_xlabel('Mean score'); axes[0].set_title('PPC: Mean Score (EIRT)'); axes[0].legend()

obs_item = Y.mean(axis=0)
rep_item = (y_rep_samples - 1).reshape(-1, N, I).mean(axis=1)
axes[1].fill_between(range(I), np.percentile(rep_item, 2, axis=0),
                     np.percentile(rep_item, 98, axis=0),
                     alpha=0.3, color='steelblue', label='96% PI')
axes[1].plot(range(I), rep_item.mean(axis=0), 'b-o', markersize=5, label='Predicted mean')
axes[1].plot(range(I), obs_item, 'r-s', markersize=7, label='Observed')
axes[1].set_xlabel('Item Index'); axes[1].set_ylabel('Mean Score')
axes[1].set_title('PPC: Item Mean Scores (96% PI)'); axes[1].legend()

fig.suptitle('PPC: EIRT Model', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
bp = (rep_means >= obs_mean).mean()
print(f"Bayesian p-value: {bp:.3f}")


## 10. True vs Posterior Mean + Wright Map

In [ ]:
theta_pm = theta_samples.mean(axis=0)
beta_pm  = beta_samples.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (true, samps, pm, col, title) in zip(axes, [
    (theta_true, theta_samples, theta_pm, 'steelblue', 'Person Ability theta'),
    (beta_true,  beta_samples,  beta_pm,  'coral',     'Item Difficulty beta'),
]):
    ci_lo = np.percentile(samps, 2, axis=0)
    ci_hi = np.percentile(samps, 98, axis=0)
    xs = true[np.argsort(true)]
    ax.fill_between(xs, ci_lo[np.argsort(true)], ci_hi[np.argsort(true)],
                    alpha=0.2, color=col, label='96% CI')
    ax.scatter(true, pm, color=col, s=20, zorder=5, alpha=0.7)
    lo, hi = min(true.min(), pm.min())-0.3, max(true.max(), pm.max())+0.3
    ax.plot([lo, hi], [lo, hi], 'k--', label='y=x')
    r = np.corrcoef(true, pm)[0,1]
    rmse = np.sqrt(np.mean((true - pm)**2))
    ax.set_xlabel('True'); ax.set_ylabel('Posterior mean')
    ax.set_title(f'{title}\nr={r:.3f}, RMSE={rmse:.3f}', fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('EIRT: True vs Posterior Mean (96% CI bands)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Wright Map
fig = plt.figure(figsize=(10, 9))
gs  = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1.0], wspace=0.05)
ax_p = fig.add_subplot(gs[0]); ax_i = fig.add_subplot(gs[1])
y_range = (-4, 4)

ax_p.hist(theta_pm, bins=20, orientation='horizontal',
          color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_range); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency'); ax_p.set_ylabel('Logit Scale')
ax_p.set_title('Person Abilities'); ax_p.axhline(0, color='gray', linestyle='--', alpha=0.5)

tau_pm = fit.stan_variable('tau').mean(axis=0)
step_colors = ['#2196F3', '#FF5722', '#4CAF50']
for i in range(I):
    b = beta_pm[i]
    ax_i.plot([0.05, 0.35], [b, b], color='coral', linewidth=2.5)
    ax_i.text(0.38, b, f'I{i+1}', fontsize=8, va='center', color='darkred')
    for k in range(K-1):
        sl = b + tau_pm[i, k]
        ax_i.plot([0.45, 0.7], [sl, sl], color=step_colors[k], linewidth=1.3, alpha=0.7)

ax_i.set_ylim(y_range); ax_i.set_xlim(0, 1.0); ax_i.set_xticks([])
ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('Item Difficulties')
ax_i.axhline(0, color='gray', linestyle='--', alpha=0.5)

fig.suptitle('Wright Map --- EIRT (PCM with Explanatory Variables)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 11. Uncertainty Analysis (96% Credible Intervals)

In [ ]:
# CI width for structural effects
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

alpha_ci_w = np.percentile(alpha_samples, 98, axis=0) - np.percentile(alpha_samples, 2, axis=0)
gamma_ci_w = np.percentile(gamma_samples, 98, axis=0) - np.percentile(gamma_samples, 2, axis=0)

axes[0].bar(range(Q), alpha_ci_w, color='steelblue', alpha=0.75, edgecolor='white')
axes[0].set_xticks(range(Q)); axes[0].set_xticklabels(alpha_labels, fontsize=9)
axes[0].set_ylabel('96% CI Width'); axes[0].set_title('Alpha: CI Width (person effects)')

axes[1].bar(range(P), gamma_ci_w, color='coral', alpha=0.75, edgecolor='white')
axes[1].set_xticks(range(P)); axes[1].set_xticklabels(gamma_labels, fontsize=9)
axes[1].set_ylabel('96% CI Width'); axes[1].set_title('Gamma: CI Width (item effects)')

plt.suptitle('Structural Effect Uncertainty (96% CI Width)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Residual variances
sigma_zeta_s = fit.stan_variable('sigma_zeta')
sigma_eps_s  = fit.stan_variable('sigma_epsilon')
print(f"sigma_zeta:   mean={sigma_zeta_s.mean():.3f} (true={sigma_zeta_true})")
print(f"sigma_epsilon: mean={sigma_eps_s.mean():.3f} (true={sigma_epsilon_true})")
print("\nProportion of theta variance explained by Z covariates:")
theta_var_total = theta_true.var()
theta_var_zeta  = sigma_zeta_true**2
print(f"  R^2 = 1 - sigma_zeta^2/Var(theta) = {1 - theta_var_zeta/theta_var_total:.3f}")


## 12. References / 참고 문헌### 국제 학술지 (확인된 논문)1. **Kim, J., & Wilson, M. (2020)**. Polytomous Item Explanatory Item Response Theory Models. *Educational and Psychological Measurement, 80*(4), 726–755. https://doi.org/10.1177/00131644198926672. **Bürkner, P.-C. (2021)**. Bayesian Item Response Modeling in R with brms and Stan. *Journal of Statistical Software, 100*(5), 1–54. https://doi.org/10.18637/jss.v100.i053. **De Boeck, P., & Wilson, M. (Eds.) (2004)**. Explanatory Item Response Models: A Generalized Linear and Nonlinear Approach. Springer. *(EIRT 이론 원전)*4. **Gelman, A., Vehtari, A., Simpson, D., Margossian, C. C., Carpenter, B., Yao, Y., Kennedy, L., Gabry, J., Bürkner, P.-C., & Modrák, M. (2020)**. Bayesian Workflow. *arXiv:2011.01808*.5. **Vehtari, A., Gelman, A., Simpson, D., Carpenter, B., & Bürkner, P.-C. (2021)**. Rank-Normalization, Folding, and Localization: An Improved R-hat for Assessing Convergence of MCMC. *Bayesian Analysis, 16*(2), 667–718.---### KCI 등재 학술지6. **한국어 학습자의 쓰기 수행 평가 신뢰도 분석 — 다국면 라쉬 모형을 사용하여** (KCI 논문번호: ART002387352). KCI 포털(www.kci.go.kr)에서 전문 검색 가능.7. **문항반응이론을 활용한 학문 목적 한국어 말하기 평가 문항 및 채점자 특성 분석** (KCI 논문번호: ART002419170). KCI 포털에서 검색 가능.8. **Rasch 모형을 적용한 문항분석 및 차별기능문항 탐색** (KCI 논문번호: ART002889793). KCI 포털에서 검색 가능.---> **KCI 검색 방법**: www.kci.go.kr → 논문검색> 추천 검색어: `설명적 IRT`, `LLTM`, `선형 로지스틱 테스트 모형`, `한국어 능력 평가`, `다층 측정 모형`